# Lab 4 Parte 2 — LSTM para monitoreo estructural (SHM)

Este notebook entrena una **LSTM** sobre el dataset de monitoreo de salud estructural (SHM) usado en los Labs 1 y 3, explotando esta vez el **orden temporal** de las lecturas de sensores. Cubrimos: preparación de ventanas deslizantes, clasificación de `Condition Label`, y dos pruebas de pronóstico — **interpolación** (rellenar un hueco interior) y **extrapolación** (predecir pasos futuros) — que son la base de cualquier sistema de alerta temprana basado en sensores.

**Secciones:**
1. Panorama RNN/LSTM en monitoreo estructural
2. Carga y orden temporal
3. Calidad de datos y balance de clases
4. Serie temporal global (EDA)
5. Series por condición estructural
6. Ventanas deslizantes y split temporal
7. Arquitectura LSTM
8. Entrenamiento y métricas
9. Pronóstico: interpolación y extrapolación (con umbral de alerta)

Cierra con una sección de reflexión para discutir con tu propio criterio de ingeniería.


In [ ]:
import sys
from pathlib import Path

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Entorno listo | device={device}")


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración en tres ejes |
| Strain | με | Deformación (extensómetro) |
| Temp | °C | Temperatura del sensor |
| **Condition Label** | **0 / 1 / 2** | **Saludable / daño menor / severo** |

Mismo CSV que **Lab 1** y **Lab 3**. Aquí explotamos el **orden temporal** con LSTM.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## 1. Panorama RNN/LSTM en monitoreo estructural

Las **RNN/LSTM** procesan **secuencias** de sensores — la memoria temporal ayuda a detectar patrones de daño que se desarrollan a lo largo del tiempo, algo que un modelo tabular como el XGBoost del Lab 3 no puede ver directamente porque trata cada fila de forma independiente.

Una **ventana deslizante** son los últimos *W* segundos de lecturas usados como entrada a la red — la etiqueta a predecir es el estado en el último instante de esa ventana. Es clave no barajar el tiempo al construir train/val: hacerlo rompería la causalidad y dejaría que el modelo "vea" información del futuro durante el entrenamiento (fuga temporal).

In [ ]:
COMPONENTES_RNN = ["ventana temporal", "LSTM", "hidden size", "dropout", "fully connected"]
print("Componentes RNN del laboratorio:")
for c in COMPONENTES_RNN:
    print(f"  · {c}")


## 2. Carga y orden temporal

El dataset tiene 5 features de sensor más `Timestamp` y la etiqueta `Condition Label`. A diferencia de los Labs 1 y 3, aquí el orden cronológico es obligatorio: lo primero que hacemos tras cargar es ordenar por `Timestamp`.

In [ ]:
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
if not RUTA_DATOS.is_file():
    # Primera ejecución: el CSV viene comprimido en data/archive.zip
    import zipfile
    with zipfile.ZipFile("data/archive.zip") as zf:
        zf.extract(RUTA_DATOS.name, "data")
    print("📦 data/archive.zip descomprimido.")
df = pd.read_csv(RUTA_DATOS)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} × {df.shape[1]}")


In [ ]:
FEATURES = [
    "Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)",
    "Strain (με)", "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features: {FEATURES}")
display(df.head(N_FILAS_HEAD))


## 3. Calidad de datos y balance de clases

Revisamos cuántas filas se pierden al eliminar nulos y cómo se distribuye `Condition Label`. `Strain` suele ser el sensor más crítico en SHM porque responde directamente a la deformación de la estructura.

In [ ]:
n_antes = len(df)
df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
print(f"Tras dropna: {n_antes} → {n_despues}")
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(conteo).plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Distribución de Condition Label')
plt.tight_layout()
plt.show()


In [ ]:
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas «{COLUMNA_REVISAR}» (crudo):")
display(stats_col)


## 4. Serie temporal global (EDA)

Antes de entrenar, visualizamos cómo evoluciona `Strain` en el tiempo — a menudo hay picos de amplitud que anteceden cambios de etiqueta, y ruido o tendencias lentas (por ejemplo, ligadas a temperatura) que conviene identificar antes de modelar.

In [ ]:
SENSOR_EDA = "Strain (με)"
N_PUNTOS_PLOT = 300
serie = df_limpio[SENSOR_EDA].iloc[:N_PUNTOS_PLOT]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(serie.values, color='#2980b9', linewidth=0.8)
ax.set_title(f'Serie temporal — {SENSOR_EDA}')
ax.set_xlabel('Índice temporal (1 Hz)'); ax.set_ylabel(SENSOR_EDA)
plt.tight_layout()
plt.show()


## 5. Series por condición estructural

Comparamos `Strain` y `Temp` entre las tres clases de `Condition Label`. La clase 2 (severo) suele mostrar mayor variabilidad en `Strain` — en un despliegue real, esto sugiere complementar con otro sensor (por ejemplo aceleración) para confirmar una alerta antes de actuar.

In [ ]:
SENSORES_COMPARAR = ["Strain (με)", "Temp (°C)"]
fig, axes = plt.subplots(1, len(SENSORES_COMPARAR), figsize=(10, 4))
if len(SENSORES_COMPARAR) == 1:
    axes = [axes]
colores = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
for ax, sensor in zip(axes, SENSORES_COMPARAR):
    for clase in [0, 1, 2]:
        sub = df_limpio[df_limpio['Condition Label'] == clase][sensor].iloc[:100]
        ax.plot(sub.values, label=f'Clase {clase}', color=colores[clase], alpha=0.8)
    ax.set_title(sensor); ax.legend(fontsize=8)
plt.suptitle('Sensores por Condition Label (primeros 100 puntos/clase)')
plt.tight_layout()
plt.show()


## 6. Ventanas deslizantes y split temporal

Construimos ventanas de `WINDOW_SIZE` lecturas consecutivas; la etiqueta a predecir es el `Condition Label` en el último instante de cada ventana. El split train/val toma el primer 80% de las ventanas como entrenamiento y el resto como validación **sin barajar** — así el modelo nunca entrena con datos posteriores a los que valida.

In [ ]:
def make_sequence_loaders(df_seq, features, window_size, batch_size):
    X_raw = df_seq[features].values
    y_raw = df_seq['Condition Label'].values.astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    seqs, labels = [], []
    for i in range(window_size, len(X_scaled)):
        seqs.append(X_scaled[i - window_size : i])
        labels.append(y_raw[i])
    X_arr = np.array(seqs, dtype=np.float32)
    y_arr = np.array(labels, dtype=np.int64)
    cut = int(0.8 * len(X_arr))
    X_train, X_val = X_arr[:cut], X_arr[cut:]
    y_train, y_val = y_arr[:cut], y_arr[cut:]
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, len(X_train), len(X_val)

print("✅ make_sequence_loaders listo (split 80/20 temporal).")


In [ ]:
WINDOW_SIZE = 30
BATCH_SIZE = 32
train_loader, val_loader, n_train, n_val = make_sequence_loaders(
    df_limpio, FEATURES, WINDOW_SIZE, BATCH_SIZE)
xb, yb = next(iter(train_loader))
print(f"Ventanas train={n_train} val={n_val} | batch X={tuple(xb.shape)} y={tuple(yb.shape)}")


## 7. Arquitectura LSTM

Una LSTM de una capa (`batch_first=True`) procesa cada ventana y nos quedamos con el **último estado oculto** — el que mejor resume la ventana reciente — para clasificar entre las 3 clases con una capa lineal.

In [ ]:
HIDDEN_SIZE = 64
N_LAYERS = 1
DROPOUT = 0.2

class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=len(FEATURES), hidden_size=HIDDEN_SIZE,
            num_layers=N_LAYERS, batch_first=True,
            dropout=DROPOUT if N_LAYERS > 1 else 0.0,
        )
        self.fc = nn.Linear(HIDDEN_SIZE, 3)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

modelo = LSTMClassifier().to(device)
print(modelo)


## 8. Entrenamiento y métricas

Entrenamos con `CrossEntropyLoss` + Adam durante `N_EPOCHS` épocas, monitoreando accuracy de train y validación en cada una. La clase 2 (severo) suele ser la más difícil de detectar por ser minoritaria.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, dev):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return loss_sum / total, correct / total

def eval_epoch(model, loader, criterion, dev):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")


In [ ]:
N_EPOCHS = 5
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
for epoch in range(N_EPOCHS):
    tl, ta = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
    vl, va = eval_epoch(modelo, val_loader, criterion, device)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    print(f"Época {epoch+1}/{N_EPOCHS} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}")
acc_val = history['val_acc'][-1]
print(f"✅ Entrenamiento completado | acc_val={acc_val:.3f}")


## 9. Pronóstico: interpolación, extrapolación y umbral de alerta

Más allá de clasificar el estado actual, entrenamos un segundo modelo — una LSTM regresora sobre `Strain` normalizado — para dos pruebas de pronóstico que son la base de cualquier sistema de alerta temprana:

- **Interpolación**: rellenar un hueco interior de la serie (valores enmascarados) y comparar con la señal real. Es la prueba más fácil, porque el modelo tiene contexto a ambos lados del hueco.
- **Extrapolación**: predecir pasos **futuros** usando solo el histórico — más exigente, porque el error típicamente crece con el horizonte de predicción.

In [ ]:
strain_raw = df_limpio['Strain (με)'].values.astype(np.float32)
strain_norm = (strain_raw - strain_raw.mean()) / (strain_raw.std() + 1e-8)
SEGMENTO_LEN = 120
W_EXTRAP = 80

class StrainLSTM(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)

def _train_strain_regressor(serie, seg_len=SEGMENTO_LEN, epochs=20):
    """Autoencoder secuencial: reconstruye tramos de Strain (base para interp/extrap)."""
    chunks = []
    for start in range(0, len(serie) - seg_len - 1, 8):
        chunks.append(serie[start : start + seg_len])
    X = torch.tensor(np.array(chunks)[:, :, None], dtype=torch.float32)
    model = StrainLSTM().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(X.to(device))
        loss = loss_fn(pred, X.to(device))
        loss.backward()
        opt.step()
    return model.eval()

modelo_reg = _train_strain_regressor(strain_norm)

def evaluar_interpolacion(model, serie, gap_inicio, gap_fin, seg_len=SEGMENTO_LEN):
    seg = serie[200 : 200 + seg_len].copy()
    entrada = seg.copy()
    entrada[gap_inicio:gap_fin] = 0.0
    x = torch.tensor(entrada[:, None], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).squeeze().cpu().numpy()
    real = seg[gap_inicio:gap_fin]
    pred_gap = pred[gap_inicio:gap_fin]
    mae = float(np.mean(np.abs(real - pred_gap)))
    return mae, real, pred_gap

def evaluar_extrapolacion(model, serie, w, horizonte, start=300):
    """Pronóstico: histórico en entrada; ceros en tramo futuro; predice H pasos."""
    full_len = w + horizonte
    hist = serie[start : start + w]
    real_fut = serie[start + w : start + w + horizonte]
    entrada = np.zeros(full_len, dtype=np.float32)
    entrada[:w] = hist
    x = torch.tensor(entrada[:, None], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).squeeze().cpu().numpy()
    pred_fut = pred[w : w + horizonte]
    mae = float(np.mean(np.abs(real_fut - pred_fut)))
    return mae, hist, real_fut, pred_fut

print("✅ modelo_reg entrenado | helpers interpolación/extrapolación listos.")


### Interpolación

Enmascaramos un tramo interior de la serie y comparamos la reconstrucción del modelo contra el valor real.

In [ ]:
GAP_INICIO = 40
GAP_FIN = 60
mae_interp, y_real, y_pred = evaluar_interpolacion(modelo_reg, strain_norm, GAP_INICIO, GAP_FIN)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(GAP_INICIO, GAP_FIN), y_real, 'o-', label='Real', color='#2c3e50')
ax.plot(range(GAP_INICIO, GAP_FIN), y_pred, 's--', label='Predicho', color='#e74c3c')
ax.set_title('Interpolación — hueco en Strain (normalizado)')
ax.legend(); plt.tight_layout(); plt.show()
print(f"MAE interpolación: {mae_interp:.4f}")


### Extrapolación y umbral de alerta

Ahora pronosticamos `HORIZONTE_EXTRAP` pasos **hacia adelante** usando solo el histórico — sin ver el futuro real. Superponemos una línea de **umbral ilustrativo** sobre el Strain normalizado pronosticado: si el pronóstico la cruza, el notebook imprime una alerta ⚠️.

El valor exacto de este umbral es **ilustrativo** — aquí no tenemos a mano la normativa estructural real. En el **Lab 5** recuperarás con RAG el valor exacto desde las normas peruanas (E.020/E.030/E.050) en PDF y podrás reemplazar este número por uno verificado contra la fuente, citando la página exacta.

In [ ]:
HORIZONTE_EXTRAP = 15
UMBRAL_ALERTA_STRAIN = 1.8  # ilustrativo, en unidades de Strain normalizado (ver Lab 5 para el valor real de norma)

mae_extrap, hist, real_fut, pred_fut = evaluar_extrapolacion(
    modelo_reg, strain_norm, W_EXTRAP, HORIZONTE_EXTRAP)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(hist)), hist, label='Histórico', color='#3498db')
off = len(hist)
ax.plot(range(off, off + len(real_fut)), real_fut, 'o-', label='Real futuro', color='#2c3e50')
ax.plot(range(off, off + len(pred_fut)), pred_fut, 's--', label='Predicho', color='#e74c3c')
ax.axvline(off - 0.5, color='gray', linestyle=':', label='Inicio forecast')
ax.axhline(UMBRAL_ALERTA_STRAIN, color='#c0392b', linestyle='-.', linewidth=1, label='Umbral ilustrativo')
ax.set_title('Extrapolación — forecast Strain (normalizado) vs. umbral de alerta')
ax.legend(); plt.tight_layout(); plt.show()
print(f"MAE extrapolación: {mae_extrap:.4f}")

if np.any(pred_fut > UMBRAL_ALERTA_STRAIN):
    paso = int(np.argmax(pred_fut > UMBRAL_ALERTA_STRAIN))
    print(f"⚠️  El pronóstico cruza el umbral ilustrativo en el paso +{paso + 1} del horizonte.")
    print("    Verifica el valor real de norma en el Lab 5 antes de usar esto como alerta operativa.")
else:
    print("✅ El pronóstico se mantiene por debajo del umbral ilustrativo en todo el horizonte.")


## Reflexión: preguntas que los alumnos necesitarían

- ¿Cuándo elegirías LSTM sobre XGBoost (Lab 3) en un edificio instrumentado, y cuándo al revés?
- El split debe ser estrictamente temporal (sin barajar). ¿Qué error concreto se filtraría en las métricas si se barajara?
- El pronóstico por extrapolación tiene un MAE mayor que el de interpolación. ¿Confiarías en él, sin más, para decidir el cierre de un puente? ¿Qué evidencia adicional pedirías?
- El umbral usado en la sección 9 es ilustrativo, no normativo. ¿Qué consecuencias tendría desplegar una alerta automática basada en un umbral no verificado contra la norma real?
- Si el sensor de Strain empezara a derivar (drift) con el tiempo, ¿cómo lo notarías en este pipeline, y en qué sección fallaría primero?
